In [0]:
%pip install geopandas rasterio shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 63.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
import tempfile
import shutil
import requests


In [0]:
import tempfile
import shutil

# Retrieve the list of flights that still need plot cropping.
# This value is passed from the '3_orquestator' task via job task values.
flight_list = dbutils.jobs.taskValues.get(
    taskKey="3_orquestator",
    key="missing_clips",
    default=[]
)

# If there is nothing to process, exit the notebook gracefully.
if not flight_list:
    dbutils.notebook.exit("No pending flights to process. Exiting gracefully.")

print(f" Received {len(flight_list)} flights for plot cropping.\n")

# ======================================================================
# MAIN LOOP: crop plots for each flight.
# ======================================================================
for flight_path in flight_list:
    print("-" * 60)
    print(f" PROCESSING FLIGHT: {flight_path}")

    # DATABRICKS FIX: convert 'dbfs:/' prefix into '/dbfs/' so the standard
    # libraries (os / glob / rasterio) can read the path correctly.
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")

    # Resolve key directories relative to the flight path:
    #   base_dir        -> the flight's base folder
    #   parent_dir      -> one level up from base_dir
    #   field_data_dir  -> where the plot geometries (vectors) live
    base_dir = os.path.dirname(flight_path)
    parent_dir = os.path.dirname(base_dir)
    field_data_dir = os.path.join(parent_dir, "field_data")

    # ------------------------------------------------------------------
    # Locate the orthomosaic (RGB.tif or MS.tif) using glob.
    # The '*/*/*' wildcards match the orthomosaic/date/sensor folder tree.
    # IMPORTANT: store matches[0] (the real resolved path), not the pattern,
    # otherwise rasterio.open() would fail on a path containing wildcards.
    # ------------------------------------------------------------------
    possible_ortho_names = ["RGB.tif", "MS.tif"]
    rgb_path = None

    for name in possible_ortho_names:
        candidate = f"{base_dir}/*/*/*/{name}"
        matches = glob.glob(candidate)
        if matches:
            rgb_path = matches[0]   # Real resolved path (no wildcards).
            break

    # Output folder where the cropped plots will be stored.
    out_dir = os.path.join(base_dir, "plot_clipped")

    # Search for the vector geometry file (.shp or .geojson) in field_data.
    print(f" Searching geometries in: {field_data_dir}")
    vector_files = glob.glob(os.path.join(field_data_dir, "*.shp")) + glob.glob(os.path.join(field_data_dir, "*.geojson"))

    # Skip this flight if no orthomosaic was found.
    if rgb_path is None:
        print(f" Error: No RGB.tif or MS.tif found in {base_dir}. Skipping flight...")
        continue
    else:
        print(f" Orthomosaic found: {os.path.basename(rgb_path)}")

    # Skip this flight if no vector geometry file was found.
    if not vector_files:
        print(f" Error: No .shp or .geojson file found in {field_data_dir}. Skipping flight...")
        continue

    # Use the first vector file found.
    vector_path = vector_files[0]
    print(f" Vector file found: {os.path.basename(vector_path)}")

    # Make sure the output directory exists.
    os.makedirs(out_dir, exist_ok=True)

    # Local scratch directory on the cluster's own disk (not the Volume).
    # Writing rasters locally first avoids seek/IO errors on the Volume.
    local_tmp_dir = tempfile.mkdtemp(prefix="plot_clip_")

    try:
        # --------------------------------------------------------------
        # Load the plot geometries and align their CRS with the raster.
        # --------------------------------------------------------------
        print(" Loading geometries and checking Coordinate Reference Systems (CRS)...")
        gdf_plots = gpd.read_file(vector_path)

        with rasterio.open(rgb_path) as src:
            raster_crs = src.crs

            # Reproject the polygons if their CRS differs from the raster's.
            if gdf_plots.crs != raster_crs:
                print(f" Reprojecting polygons from {gdf_plots.crs} to {raster_crs}...")
                gdf_plots = gdf_plots.to_crs(raster_crs)

            # ----------------------------------------------------------
            # Choose the output format based on the number of bands:
            #   <= 4 bands  -> PNG (standard RGB/RGBA imagery)
            #   >  4 bands  -> GeoTIFF (multispectral data)
            # ----------------------------------------------------------
            n_bands = src.count
            if n_bands <= 4:
                out_driver = "PNG"
                out_ext = "png"
            else:
                out_driver = "GTiff"
                out_ext = "tif"
            print(f" Detected {n_bands} bands -> using {out_driver} for output ({out_ext}).")

            # ----------------------------------------------------------
            # Crop the orthomosaic once per plot geometry.
            # ----------------------------------------------------------
            print(f" Clipping {len(gdf_plots)} detected plots...")

            for idx, row in gdf_plots.iterrows():
                # Convert the geometry to the mapping format rasterio expects.
                geometry = [mapping(row.geometry)]

                # Mask (clip) the raster to the plot geometry and crop tightly.
                out_image, out_transform = mask(src, geometry, crop=True)

                # Copy the source metadata and update it for the clipped output.
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": out_driver,
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
                })

                # One output file per plot, numbered starting at 1.
                file_name = f"Plot_{idx + 1}.{out_ext}"

                # Write to LOCAL disk first (avoids seek errors on the Volume).
                local_out_path = os.path.join(local_tmp_dir, file_name)
                with rasterio.open(local_out_path, "w", **out_meta) as dest:
                    dest.write(out_image)

                # Then copy the finished file to the Volume.
                final_out_path = os.path.join(out_dir, file_name)
                shutil.copyfile(local_out_path, final_out_path)

            print(f" SUCCESS: {len(gdf_plots)} images saved to {out_dir}")

    except Exception as e:
        # Catch errors per flight so the loop continues with the rest.
        print(f" An error occurred processing this flight: {e}")

    finally:
        # Clean up local scratch files regardless of success/failure.
        shutil.rmtree(local_tmp_dir, ignore_errors=True)

print("\n" + "="*70)
print(" PLOT CROPPING PIPELINE FINISHED SUCCESSFULLY.")

In [0]:
# ── Job ID to trigger ────────────────────────────────────────────────────
# This is the Databricks Job ID of "PHENO_I" (Job 4). Once this notebook's
# own work is done, it will kick off that job to run the phenotyping
# analysis on the newly clipped plots.
JOB_4_ID = 713354967172258 #Change
 
# ── Get the current workspace context (host + auth token) ───────────────
# This lets the notebook call the Databricks REST API on its own, using
# the same workspace/credentials it's currently running in — no need to
# hardcode a host URL or token.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
 
# ── Build the API request to trigger the job ─────────────────────────────
# "run-now" starts an existing job immediately (equivalent to clicking
# "Run now" in the Databricks UI), using the job_id defined above.
url = f"{host}/api/2.1/jobs/run-now"
headers = {"Authorization": f"Bearer {token}"}
data = {"job_id": JOB_4_ID}
 
# ── Send the request and report the outcome ──────────────────────────────
response = requests.post(url, headers=headers, json=data)
 
if response.status_code == 200:
    # A 200 response means Databricks accepted the request and started the run.
    print(" Trigger successful! PhenoI Job has been started.")
else:
    # Anything else means the trigger failed — print the API's error message
    # so it's clear what went wrong (e.g. wrong job ID, permissions issue).
    print(f" Error triggering Job: {response.text}")